 **Smart Logistics Data Engineering & Real-Time Monitoring**

This notebook prepares delivery-event data through the **Silver** and **Gold** layers using PySpark in Microsoft Fabric.

Pipeline: Source CSV → Silver Delta table → Gold performance tables


## 1. Imports and source configuration

In [1]:
from pyspark.sql import functions as F

SOURCE_PATH = "Files/delivery_events.csv"
SILVER_TABLE = "silver_delivery_events"


StatementMeta(, ee0f6f4a-7533-49b2-a265-3493ae247e68, 3, Finished, Available, Finished, False)

## 2. Read the source data

In [2]:
delivery_events_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(SOURCE_PATH)
)

print("Source row count:", delivery_events_df.count())
delivery_events_df.printSchema()
display(delivery_events_df.limit(10))


StatementMeta(, ee0f6f4a-7533-49b2-a265-3493ae247e68, 4, Finished, Available, Finished, False)

Source row count: 170820
root
 |-- event_id: string (nullable = true)
 |-- load_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- facility_id: string (nullable = true)
 |-- scheduled_datetime: timestamp (nullable = true)
 |-- actual_datetime: timestamp (nullable = true)
 |-- detention_minutes: integer (nullable = true)
 |-- on_time_flag: boolean (nullable = true)
 |-- location_city: string (nullable = true)
 |-- location_state: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 03fdef6f-18c0-4fff-a9a6-8fb8aab3bbec)

## 3. Initial data quality and profiling

In [3]:
# Check null values across all source columns.
source_null_counts = delivery_events_df.select(
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in delivery_events_df.columns
    ]
)
display(source_null_counts)


StatementMeta(, ee0f6f4a-7533-49b2-a265-3493ae247e68, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 708725ef-b528-4c8d-8a44-d13754fe6129)

In [4]:
# Check exact duplicate rows and event-type distribution.
source_row_count = delivery_events_df.count()
unique_row_count = delivery_events_df.dropDuplicates().count()

print("Total rows:", source_row_count)
print("Unique rows:", unique_row_count)
print("Duplicate rows:", source_row_count - unique_row_count)

display(
    delivery_events_df
    .groupBy("event_type")
    .count()
    .orderBy("event_type")
)


StatementMeta(, ee0f6f4a-7533-49b2-a265-3493ae247e68, 6, Finished, Available, Finished, False)

Total rows: 170820
Unique rows: 170820
Duplicate rows: 0


SynapseWidget(Synapse.DataFrame, 4539d6f1-9730-45f6-9a2a-52fc7bc8b8b2)

In [5]:
# Inspect the date range represented in the source.
display(
    delivery_events_df.agg(
        F.min("scheduled_datetime").alias("earliest_scheduled"),
        F.max("scheduled_datetime").alias("latest_scheduled"),
        F.min("actual_datetime").alias("earliest_actual"),
        F.max("actual_datetime").alias("latest_actual")
    )
)


StatementMeta(, ee0f6f4a-7533-49b2-a265-3493ae247e68, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9ab1689e-8c4b-4ed6-a694-c617676e81ef)

## 4. Calculate delay minutes and validate timing flags

In [6]:
# Positive delay = late; negative delay = early; zero = exactly on schedule.
delivery_events_delay_df = delivery_events_df.withColumn(
    "delay_minutes",
    (
        F.unix_timestamp("actual_datetime")
        - F.unix_timestamp("scheduled_datetime")
    ) / 60
)

display(
    delivery_events_delay_df.select(
        "event_id",
        "event_type",
        "scheduled_datetime",
        "actual_datetime",
        "delay_minutes",
        "on_time_flag"
    ).limit(20)
)


StatementMeta(, ee0f6f4a-7533-49b2-a265-3493ae247e68, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4d2fd20b-041a-4a6f-8c87-ce62aeb4681e)

In [7]:
# Compare the source on_time_flag with a timestamp-based check.
# This is a validation view; the source on_time_flag is retained for reporting.
validation_df = delivery_events_delay_df.withColumn(
    "calculated_on_time",
    F.col("delay_minutes") <= 0
)

display(
    validation_df
    .groupBy("on_time_flag", "calculated_on_time")
    .count()
    .orderBy("on_time_flag", "calculated_on_time")
)


StatementMeta(, ee0f6f4a-7533-49b2-a265-3493ae247e68, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4a3f78a5-1c8c-4499-a600-76f2592e2c29)

In [8]:
# Review delay ranges for each source on_time_flag value.
display(
    validation_df
    .groupBy("on_time_flag")
    .agg(
        F.min("delay_minutes").alias("minimum_delay"),
        F.max("delay_minutes").alias("maximum_delay"),
        F.count("*").alias("row_count")
    )
)


StatementMeta(, ee0f6f4a-7533-49b2-a265-3493ae247e68, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6cd9330e-1135-47f2-926a-bcc87c6d1a86)

## 5. Build and validate the Silver layer

The Silver table keeps the source event attributes and the calculated `delay_minutes` field. The temporary `calculated_on_time` validation column is removed. The original `on_time_flag` is preserved and used consistently in the Gold aggregations.


In [9]:
silver_delivery_events_df = validation_df.drop("calculated_on_time")

print("Silver row count:", silver_delivery_events_df.count())

duplicate_event_ids = (
    silver_delivery_events_df
    .groupBy("event_id")
    .count()
    .filter(F.col("count") > 1)
)
print("Duplicate event IDs:", duplicate_event_ids.count())


StatementMeta(, ee0f6f4a-7533-49b2-a265-3493ae247e68, 11, Finished, Available, Finished, False)

Silver row count: 170820
Duplicate event IDs: 0


In [10]:
# Check nulls in the key Silver columns.
key_columns = [
    "event_id",
    "load_id",
    "trip_id",
    "event_type",
    "facility_id",
    "scheduled_datetime",
    "actual_datetime",
    "delay_minutes"
]

display(
    silver_delivery_events_df.select(
        *[
            F.sum(F.col(c).isNull().cast("int")).alias(c)
            for c in key_columns
        ]
    )
)


StatementMeta(, ee0f6f4a-7533-49b2-a265-3493ae247e68, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9056da34-c0be-45b6-ae79-efb1df0dc51b)

In [11]:
# Save the Silver Delta table.
(
    silver_delivery_events_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)

silver_df = spark.table(SILVER_TABLE)
print("Saved Silver rows:", silver_df.count())
silver_df.printSchema()
display(silver_df.limit(10))


StatementMeta(, ee0f6f4a-7533-49b2-a265-3493ae247e68, 13, Finished, Available, Finished, False)

Saved Silver rows: 170820
root
 |-- event_id: string (nullable = true)
 |-- load_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- facility_id: string (nullable = true)
 |-- scheduled_datetime: timestamp (nullable = true)
 |-- actual_datetime: timestamp (nullable = true)
 |-- detention_minutes: integer (nullable = true)
 |-- on_time_flag: boolean (nullable = true)
 |-- location_city: string (nullable = true)
 |-- location_state: string (nullable = true)
 |-- delay_minutes: double (nullable = true)



SynapseWidget(Synapse.DataFrame, 5168a912-566e-4a80-bcd4-76e54f6732a7)

## 6. Gold layer — delivery performance

In [12]:
gold_delivery_performance = (
    silver_df
    .groupBy("event_type")
    .agg(
        F.count("*").alias("total_events"),
        F.sum(F.when(F.col("on_time_flag") == True, 1).otherwise(0)).alias("on_time_events"),
        F.sum(F.when(F.col("on_time_flag") == False, 1).otherwise(0)).alias("late_events"),
        F.round(F.avg("delay_minutes"), 2).alias("avg_delay_minutes"),
        F.round(F.avg("detention_minutes"), 2).alias("avg_detention_minutes")
    )
    .withColumn(
        "on_time_percentage",
        F.round(F.col("on_time_events") / F.col("total_events") * 100, 2)
    )
)

display(gold_delivery_performance)


StatementMeta(, ee0f6f4a-7533-49b2-a265-3493ae247e68, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e2b05197-2fa0-419d-953b-55dcc3589495)

In [13]:
(
    gold_delivery_performance.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_delivery_performance")
)


StatementMeta(, ee0f6f4a-7533-49b2-a265-3493ae247e68, 15, Finished, Available, Finished, False)

## 7. Gold layer — facility performance

In [14]:
gold_facility_performance = (
    silver_df
    .groupBy("facility_id")
    .agg(
        F.count("*").alias("total_events"),
        F.sum(F.when(F.col("on_time_flag") == True, 1).otherwise(0)).alias("on_time_events"),
        F.sum(F.when(F.col("on_time_flag") == False, 1).otherwise(0)).alias("late_events"),
        F.round(F.avg("delay_minutes"), 2).alias("avg_delay_minutes"),
        F.round(F.avg("detention_minutes"), 2).alias("avg_detention_minutes")
    )
    .withColumn(
        "on_time_percentage",
        F.round(F.col("on_time_events") / F.col("total_events") * 100, 2)
    )
)

display(gold_facility_performance)


StatementMeta(, ee0f6f4a-7533-49b2-a265-3493ae247e68, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ddcab359-f665-4f52-ae89-a236b66e9f14)

In [15]:
(
    gold_facility_performance.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_facility_performance")
)


StatementMeta(, ee0f6f4a-7533-49b2-a265-3493ae247e68, 17, Finished, Available, Finished, False)

## 8. Gold layer — location performance

In [16]:
gold_location_performance = (
    silver_df
    .groupBy("location_city", "location_state")
    .agg(
        F.count("*").alias("total_events"),
        F.sum(F.when(F.col("on_time_flag") == True, 1).otherwise(0)).alias("on_time_events"),
        F.sum(F.when(F.col("on_time_flag") == False, 1).otherwise(0)).alias("late_events"),
        F.round(F.avg("delay_minutes"), 2).alias("avg_delay_minutes"),
        F.round(F.avg("detention_minutes"), 2).alias("avg_detention_minutes")
    )
    .withColumn(
        "on_time_percentage",
        F.round(F.col("on_time_events") / F.col("total_events") * 100, 2)
    )
)

display(gold_location_performance)


StatementMeta(, ee0f6f4a-7533-49b2-a265-3493ae247e68, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2eb9e0b6-d6e1-462c-8111-b6adffd54d43)

In [17]:
(
    gold_location_performance.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_location_performance")
)


StatementMeta(, ee0f6f4a-7533-49b2-a265-3493ae247e68, 19, Finished, Available, Finished, False)

## 9. Gold layer validation

In [18]:
gold_table_names = [
    "gold_delivery_performance",
    "gold_facility_performance",
    "gold_location_performance"
]

for table_name in gold_table_names:
    df = spark.table(table_name)
    invalid_groups = df.filter(
        (F.col("on_time_events") + F.col("late_events"))
        != F.col("total_events")
    ).count()
    invalid_percentages = df.filter(
        F.abs(
            F.col("on_time_percentage")
            - F.round(F.col("on_time_events") / F.col("total_events") * 100, 2)
        ) > 0.001
    ).count()
    print(
        table_name,
        "| rows:", df.count(),
        "| invalid event totals:", invalid_groups,
        "| invalid percentages:", invalid_percentages
    )


StatementMeta(, ee0f6f4a-7533-49b2-a265-3493ae247e68, 20, Finished, Available, Finished, False)

gold_delivery_performance | rows: 2 | invalid event totals: 0 | invalid percentages: 0
gold_facility_performance | rows: 50 | invalid event totals: 0 | invalid percentages: 0
gold_location_performance | rows: 20 | invalid event totals: 0 | invalid percentages: 0


In [19]:
# Check null values in each Gold table.
for table_name in gold_table_names:
    df = spark.table(table_name)
    null_counts = df.select(
        *[
            F.sum(F.col(c).isNull().cast("int")).alias(c)
            for c in df.columns
        ]
    )
    print(f"Null check: {table_name}")
    display(null_counts)


StatementMeta(, ee0f6f4a-7533-49b2-a265-3493ae247e68, 21, Finished, Available, Finished, False)

Null check: gold_delivery_performance


SynapseWidget(Synapse.DataFrame, b073241b-8272-4133-a3cd-6cc433beba21)

Null check: gold_facility_performance


SynapseWidget(Synapse.DataFrame, f2d7fd5e-baa1-4ea4-bde5-b5dd4e57c83a)

Null check: gold_location_performance


SynapseWidget(Synapse.DataFrame, 41ce46ed-e533-4f6b-ba8f-7df7d10f26cd)

## 10. Final table inventory and row reconciliation

In [20]:
print("Silver total rows:", silver_df.count())

for table_name in gold_table_names:
    total_events = (
        spark.table(table_name)
        .agg(F.sum("total_events").alias("total_events"))
        .first()["total_events"]
    )
    print(f"{table_name} total events:", total_events)

spark.sql("SHOW TABLES").show(truncate=False)


StatementMeta(, ee0f6f4a-7533-49b2-a265-3493ae247e68, 22, Finished, Available, Finished, False)

Silver total rows: 170820
gold_delivery_performance total events: 170820
gold_facility_performance total events: 170820
gold_location_performance total events: 170820
+----------------------------------------+-------------------------+-----------+
|namespace                               |tableName                |isTemporary|
+----------------------------------------+-------------------------+-----------+
|Smart_Logistics_DE.LH_SmartLogistics.dbo|gold_delivery_performance|false      |
|Smart_Logistics_DE.LH_SmartLogistics.dbo|gold_facility_performance|false      |
|Smart_Logistics_DE.LH_SmartLogistics.dbo|gold_location_performance|false      |
|Smart_Logistics_DE.LH_SmartLogistics.dbo|silver_delivery_events   |false      |
+----------------------------------------+-------------------------+-----------+

